# BulkFormer DX Clinical Pipeline (37M and 147M)

This notebook reproduces the clinical RNA-seq pipeline for 37M and 147M variants.

Requires:
- `data/clinical_rnaseq/raw_counts.tsv`, `gene_annotation_v29.tsv`, `sample_annotation.tsv`
- BulkFormer assets and `model/BulkFormer_37M.pt` (and `BulkFormer_147M.pt` for 147M)

In [2]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('..').resolve()
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT)

def run(cmd: list[str], cwd: Path = ROOT) -> int:
    print(' '.join(cmd))
    return subprocess.run(cmd, cwd=cwd, env=env).returncode

## 1. Preprocess (shared)

In [3]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'preprocess',
     '--counts', 'data/clinical_rnaseq/raw_counts.tsv',
     '--annotation', 'data/clinical_rnaseq/gene_annotation_v29.tsv',
     '--output-dir', 'runs/clinical_preprocess_37M',
     '--counts-orientation', 'genes-by-samples'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli preprocess --counts data/clinical_rnaseq/raw_counts.tsv --annotation data/clinical_rnaseq/gene_annotation_v29.tsv --output-dir runs/clinical_preprocess_37M --counts-orientation genes-by-samples
Wrote preprocessing outputs to runs/clinical_preprocess_37M
Samples: 146 | input genes: 60788 | BulkFormer-valid genes: 19751/20010


0

## 2. Anomaly score + calibrate (37M)

In [4]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'score',
     '--input', 'runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv',
     '--valid-gene-mask', 'runs/clinical_preprocess_37M/valid_gene_mask.tsv',
     '--output-dir', 'runs/clinical_anomaly_score_37M',
     '--variant', '37M', '--device', 'cuda', '--mc-passes', '16', '--mask-prob', '0.15'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly score --input runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv --valid-gene-mask runs/clinical_preprocess_37M/valid_gene_mask.tsv --output-dir runs/clinical_anomaly_score_37M --variant 37M --device cuda --mc-passes 16 --mask-prob 0.15


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote anomaly scoring outputs to runs/clinical_anomaly_score_37M
Samples: 146 | valid genes: 19751 | MC passes: 16 | mean cohort abs residual: 0.8603


0

In [5]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'calibrate',
     '--scores', 'runs/clinical_anomaly_score_37M',
     '--output-dir', 'runs/clinical_anomaly_calibrated_37M',
     '--alpha', '0.05'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly calibrate --scores runs/clinical_anomaly_score_37M --output-dir runs/clinical_anomaly_calibrated_37M --alpha 0.05
Wrote calibrated anomaly outputs to runs/clinical_anomaly_calibrated_37M
Samples: 146 | scored genes: 2668794 | method: none
Outlier counts: mean=79.4 | median=40 | max=1617 | inflation=0.09x


0

## 3. Anomaly score + calibrate (147M)

147M is slower; use batch-size 4, mc-passes 8. Alpha=0.01 for stricter outlier calls.

In [6]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'score',
     '--input', 'runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv',
     '--valid-gene-mask', 'runs/clinical_preprocess_37M/valid_gene_mask.tsv',
     '--output-dir', 'runs/clinical_anomaly_score_147M',
     '--variant', '147M', '--device', 'cuda', '--batch-size', '4', '--mc-passes', '8', '--mask-prob', '0.15'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly score --input runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv --valid-gene-mask runs/clinical_preprocess_37M/valid_gene_mask.tsv --output-dir runs/clinical_anomaly_score_147M --variant 147M --device cuda --batch-size 4 --mc-passes 8 --mask-prob 0.15


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote anomaly scoring outputs to runs/clinical_anomaly_score_147M
Samples: 146 | valid genes: 19751 | MC passes: 8 | mean cohort abs residual: 0.7229


0

In [7]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'calibrate',
     '--scores', 'runs/clinical_anomaly_score_147M',
     '--output-dir', 'runs/clinical_anomaly_calibrated_147M',
     '--alpha', '0.01'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly calibrate --scores runs/clinical_anomaly_score_147M --output-dir runs/clinical_anomaly_calibrated_147M --alpha 0.01
Wrote calibrated anomaly outputs to runs/clinical_anomaly_calibrated_147M
Samples: 146 | scored genes: 2097123 | method: none
Outlier counts: mean=39.6 | median=20 | max=434 | inflation=0.28x


0

## 4. Embeddings (37M and 147M)

In [8]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'embeddings', 'extract',
     '--input', 'runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv',
     '--valid-gene-mask', 'runs/clinical_preprocess_37M/valid_gene_mask.tsv',
     '--output-dir', 'runs/clinical_embeddings_37M', '--variant', '37M', '--device', 'cuda'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli embeddings extract --input runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv --valid-gene-mask runs/clinical_preprocess_37M/valid_gene_mask.tsv --output-dir runs/clinical_embeddings_37M --variant 37M --device cuda


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote runs/clinical_embeddings_37M/sample_embeddings.tsv (146 samples x 131 dims)


0

In [9]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'embeddings', 'extract',
     '--input', 'runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv',
     '--valid-gene-mask', 'runs/clinical_preprocess_37M/valid_gene_mask.tsv',
     '--output-dir', 'runs/clinical_embeddings_147M', '--variant', '147M', '--device', 'cuda', '--batch-size', '4'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli embeddings extract --input runs/clinical_preprocess_37M/aligned_log1p_tpm.tsv --valid-gene-mask runs/clinical_preprocess_37M/valid_gene_mask.tsv --output-dir runs/clinical_embeddings_147M --variant 147M --device cuda --batch-size 4


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote runs/clinical_embeddings_147M/sample_embeddings.tsv (146 samples x 643 dims)


0

## 5. Merge annotation

In [10]:
for v in ['37M', '147M']:
    run([sys.executable, 'scripts/merge_clinical_annotation.py', '--variant', v])

/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/merge_clinical_annotation.py --variant 37M
Wrote runs/clinical_annotated/cohort_scores_annotated.tsv
Wrote runs/clinical_annotated/calibration_summary_annotated.tsv
Wrote runs/clinical_annotated/sample_embeddings_annotated.tsv
/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/merge_clinical_annotation.py --variant 147M
Wrote runs/clinical_annotated_147M/cohort_scores_annotated.tsv
Wrote runs/clinical_annotated_147M/calibration_summary_annotated.tsv
Wrote runs/clinical_annotated_147M/sample_embeddings_annotated.tsv


## 6. Calibration analysis (figures)

In [11]:
for v in ['37M', '147M']:
    run([sys.executable, 'scripts/calibration_analysis.py', '--variant', v])

/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/calibration_analysis.py --variant 37M
Wrote reports/figures/calibration_outliers_per_sample_37M.tsv
Wrote 20 gene residual histograms to reports/figures/calibration_gene_histograms_37M
Wrote z-score distribution to reports/figures
Wrote reports/figures/calibration_summary_stats_37M.json
/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/calibration_analysis.py --variant 147M
Wrote reports/figures/calibration_outliers_per_sample_147M.tsv
Wrote 20 gene residual histograms to reports/figures/calibration_gene_histograms_147M
Wrote z-score distribution to reports/figures
Wrote reports/figures/calibration_summary_stats_147M.json


## 7. Generate reports

In [12]:
run([sys.executable, 'scripts/generate_clinical_report.py', '--variant', 'both'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/generate_clinical_report.py --variant both
Wrote /home/smirnov/projects/BulkFormer/reports/bulkformer_dx_clinical_report.md
Wrote /home/smirnov/projects/BulkFormer/reports/bulkformer_dx_clinical_report_147M.md


0